# Lab 4 — Seaborn Visualizations (Zomato EDA)

**Day 02 · Python for Data Science · Cisco AI/ML Training**

---

## Learning objectives

1. Create **histogram** and **box plot** visualizations with Seaborn.
2. Combine **matplotlib** figures with multiple subplots.
3. Interpret distribution shape and categorical comparisons.
4. Save figures to disk for reports and presentations.

> **Checkpoints:** `rating_distribution.png` saved · mean rating ≈ **3.70** · top avg-cost city **Kolkata**

**Companion script:** `../scripts/lab04_seaborn_plots.py`


## Why visualize before modeling?

| Question | Chart type |
|----------|------------|
| How are ratings distributed? | Histogram / KDE |
| How does cost vary by city? | Box plot |
| Are there outliers? | Box plot whiskers |
| Relationships between numeric features? | Scatter (Lab 5+) |

**Rule of thumb:** Never fit a model on data you have not plotted.


---

## 1. Imports and configuration

In **Jupyter** we use the inline backend to see plots in the notebook. The companion script uses `Agg` for headless saving — both are valid.


In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

# Resolve GH root (same pattern as Lab 3)
GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-02":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "zomato" / "zomato_restaurants.csv").is_file():
            GH_ROOT = parent
            break

ZOMATO_CSV = GH_ROOT / "data" / "zomato" / "zomato_restaurants.csv"
OUTPUT_DIR = GH_ROOT / "hands-on" / "day-02" / "scripts" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
print("Data:", ZOMATO_CSV.name)
print("Output:", OUTPUT_DIR)


---

## 2. Load data


In [ ]:
df = pd.read_csv(ZOMATO_CSV)
print(f"Loaded {df.shape[0]} rows")
assert df.shape[0] == 500


---

## 3. Histogram — distribution of `aggregate_rating`

`sns.histplot` shows frequency per bin. `kde=True` overlays a smooth density curve.

**Read the chart:**
- **Peak** ≈ where most restaurants cluster.
- **Spread** — wide = diverse ratings; narrow = consensus.
- **Skew** — tail toward low or high ratings.


In [ ]:
fig1, ax1 = plt.subplots(figsize=(6, 4))
sns.histplot(df["aggregate_rating"], bins=15, kde=True, ax=ax1, color="steelblue")
ax1.set_title("Distribution of aggregate rating")
ax1.set_xlabel("aggregate_rating")
ax1.set_ylabel("count")
plt.tight_layout()
plt.show()


### Experiment — change `bins`

Re-run the cell above with `bins=10` or `bins=25`. More bins → finer detail but noisier; fewer bins → smoother overview.


---

## 4. Box plot — cost by city (top 5)

Box plots summarize **median**, **quartiles**, and **outliers** per category.

We filter to the **five most common cities** so labels stay readable.


In [ ]:
top_cities = df["city"].value_counts().head(5).index.tolist()
city_subset = df[df["city"].isin(top_cities)]

print("Top 5 cities by restaurant count:", top_cities)

fig2, ax2 = plt.subplots(figsize=(7, 4))
sns.boxplot(data=city_subset, x="city", y="average_cost_for_two", ax=ax2, palette="pastel")
ax2.set_title("Cost for two — top 5 cities")
ax2.set_xlabel("city")
ax2.set_ylabel("average_cost_for_two")
ax2.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


**Interpretation tips:**
- **Box height** = middle 50% of costs (IQR).
- **Whiskers** extend to typical range; points beyond = outliers.
- Compare **median lines** across cities — which city is generally pricier?


---

## 5. Combined dashboard (matches lab script)

Side-by-side subplots mirror the maintainer dry-run script output.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.histplot(df["aggregate_rating"], bins=15, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution of aggregate rating")

sns.boxplot(data=city_subset, x="city", y="average_cost_for_two", ax=axes[1], palette="pastel")
axes[1].set_title("Cost for two — top 5 cities")
axes[1].tick_params(axis="x", rotation=30)

fig.tight_layout()
rating_plot = OUTPUT_DIR / "rating_distribution.png"
fig.savefig(rating_plot, dpi=100, bbox_inches="tight")
plt.show()

print(f"Saved: {rating_plot}")
assert rating_plot.is_file(), "Plot file not created"


In [ ]:
# Display saved file inline (notebook verification)
display(Image(filename=str(rating_plot)))


---

## 6. Numeric checkpoints


In [ ]:
mean_rating = df["aggregate_rating"].mean()
city_means = df.groupby("city")["average_cost_for_two"].mean().sort_values(ascending=False)
top_city = city_means.index[0]
top_city_cost = city_means.iloc[0]

print("Lab 4 — Seaborn plots")
print(f"rating plot saved: {rating_plot.name}")
print(f"mean rating: {mean_rating:.2f}")
print(f"top city by avg cost: {top_city} ({top_city_cost:.0f})")

assert abs(mean_rating - 3.70) < 0.05
assert top_city == "Kolkata"
print("\n✓ Checkpoint assertions passed")


---

## 7. Optional extension — count plot

Seaborn `countplot` quickly shows category frequencies:


In [ ]:
fig3, ax3 = plt.subplots(figsize=(6, 3))
sns.countplot(data=df, x="online_order", ax=ax3, palette="Set2")
ax3.set_title("Online order availability")
plt.tight_layout()
plt.show()


## Extension — scatter: rating vs cost (engagement)

<!-- cisco-enrich-2026-06 -->

In [ ]:
fig4, ax4 = plt.subplots(figsize=(6, 4))
sns.scatterplot(data=df, x="average_cost_for_two", y="aggregate_rating",
                hue="online_order", alpha=0.5, ax=ax4)
ax4.set_title("Rating vs cost (colored by online order)")
plt.tight_layout()
scatter_plot = OUTPUT_DIR / "rating_vs_cost_scatter.png"
fig4.savefig(scatter_plot, dpi=100)
plt.show()
print("saved:", scatter_plot.name)


## Extension — pairplot on numeric columns (top 80 rows)

In [ ]:
cols = ["aggregate_rating", "votes", "average_cost_for_two"]
sns.pairplot(df[cols].head(80), corner=True, diag_kind="hist", height=2.2)
plt.suptitle("Zomato numeric pairplot (sample)", y=1.02)
plt.show()


---

## Reflection questions

1. Is the rating distribution roughly symmetric or skewed?
2. Which city shows the widest cost spread in the box plot?
3. Why save figures to PNG instead of only showing inline?

**Previous:** [Lab 3 — Pandas Zomato load](lab03_pandas_zomato_load.ipynb)  
**Next:** [Lab 5 — Linear regression fit](lab05_linear_regression_fit.ipynb)
